# Expanding Attention

#  Mike and Adam
"""
Based on last weeks attention model.

Added quantization and sparse pattern options to the attention model.
"""

In [ ]:
import math
from typing import Optional

import torch
from torch import nn


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 10000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, D]
        T = x.size(1)
        return x + self.pe[:T, :]


class TransformerTextClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int = 4,
        d_model: int = 256,
        nhead: int = 4,
        dim_feedforward: int = 512,
        nlayers: int = 4,
        dropout: float = 0.1,
        pad_idx: int = 1,
        use_quantization: bool = False,
        use_sparse_pattern: bool = False,
        sparsity_level: float = 0.2         # fraction of weights to prune
    ):
        super().__init__()
        self.pad_idx = pad_idx
        self.use_quantization = use_quantization
        self.use_sparse_pattern = use_sparse_pattern
        self.sparsity_level = sparsity_level

        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_enc = SinusoidalPositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout,
            batch_first=True, activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=nlayers)
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

        # Apply quantization if requested
        if self.use_quantization:
            self._apply_quantization()
        # Apply use sparse if requested
        if self.use_sparse_pattern:
            self._apply_sparsity_pattern()


    def _apply_quantization(self):
        """
        Applies dynamic quantization to linear layers.
        This reduces model size and may improve inference speed on CPU.
        """
        self.encoder = torch.quantization.quantize_dynamic(
            self.encoder, {nn.Linear}, dtype=torch.qint8
        )
        self.classifier = torch.quantization.quantize_dynamic(
            self.classifier, {nn.Linear}, dtype=torch.qint8
        )

    def _apply_sparsity_pattern(self):
        """Applies structured sparsity (L1 unstructured by default) to linear layers."""
        import torch.nn.utils.prune as prune

        def prune_module(module):
            for name, submodule in module.named_modules():
                if isinstance(submodule, nn.Linear):
                    prune.l1_unstructured(submodule, name='weight', amount=self.sparsity_level)

        prune_module(self.encoder)
        prune_module(self.classifier)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T]
        key_padding_mask = (x == self.pad_idx)  # [B, T]
        h = self.token_emb(x)                   # [B, T, D]
        h = self.pos_enc(h)
        h = self.encoder(h, src_key_padding_mask=key_padding_mask)
        h = self.norm(h)
        # mask‑aware mean pooling
        lengths = (~key_padding_mask).sum(dim=1).clamp(min=1).unsqueeze(-1)
        pooled = (h * (~key_padding_mask).unsqueeze(-1)).sum(dim=1) / lengths
        logits = self.classifier(pooled)
        return logits

In [ ]:
"""
TorchText-free data loader for AG News classification.
Downloads CSVs and builds a tiny vocab/tokenizer from scratch.
"""
from typing import List, Tuple
import os, csv, re, urllib.request
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

_TRAIN_URL = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"
_TEST_URL  = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv"
_TRAIN_CSV = "/content/ag_news_train.csv"
_TEST_CSV  = "/content/ag_news_test.csv"

_TOKEN_RE = re.compile(r"[A-Za-z0-9]+|[^\w\s]")

def _basic_tokenize(text: str) -> List[str]:
    return [t.lower() for t in _TOKEN_RE.findall(text)]

def _ensure_data():
    if not os.path.exists(_TRAIN_CSV):
        urllib.request.urlretrieve(_TRAIN_URL, _TRAIN_CSV)
    if not os.path.exists(_TEST_CSV):
        urllib.request.urlretrieve(_TEST_URL, _TEST_CSV)

def _read_csv(path: str) -> List[Tuple[int, str]]:
    rows=[]
    with open(path, newline="", encoding="utf-8") as f:
        for label, title, desc in csv.reader(f):
            text=f"{title}. {desc}"
            rows.append((int(label), text))
    return rows

class _Vocab:
    def __init__(self, stoi): self._stoi=stoi
    def __len__(self): return len(self._stoi)
    def __call__(self,tokens): return [self._stoi.get(t,self._stoi["<unk>"]) for t in tokens]
    def get_stoi(self): return self._stoi

def _build_vocab(train, min_freq=2):
    from collections import Counter
    c=Counter()
    for _,text in train: c.update(_basic_tokenize(text))
    stoi={"<unk>":0,"<pad>":1}
    for tok,freq in c.items():
        if freq>=min_freq: stoi[tok]=len(stoi)
    return _Vocab(stoi), stoi["<pad>"]

def get_dataloaders(batch_size, device, min_freq=2):
    _ensure_data()
    train=_read_csv(_TRAIN_CSV)
    test=_read_csv(_TEST_CSV)
    vocab,pad_idx=_build_vocab(train,min_freq)

    def text_to_ids(t): return vocab(_basic_tokenize(t))
    def collate(batch):
        labels,seqs=[],[]
        for label,text in batch:
            labels.append(label-1)
            seqs.append(torch.tensor(text_to_ids(text)))
        padded=pad_sequence(seqs,batch_first=True,padding_value=pad_idx)
        return padded.to(device),torch.tensor(labels,device=device)
    return (
        DataLoader(train,batch_size=batch_size,shuffle=True,collate_fn=collate),
        DataLoader(test,batch_size=batch_size,shuffle=False,collate_fn=collate),
        vocab,pad_idx
    )


In [ ]:
"""
Training entrypoint for the non‑LLM Transformer text classifier.
Usage example (CPU‑friendly):

  --epochs 3 \
  --d_model 128 \
  --nhead 4 \
  --nlayers 2 \
  --ff 256 \
  --batch_size 128 \
  --lr 5e-4
"""
import argparse
import random
from typing import Tuple

import torch
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from tqdm import tqdm

def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def accuracy(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = logits.argmax(dim=-1)
    return (preds == y).float().mean().item()


def train_epoch(model, loader, criterion, optimizer, scheduler=None):
    model.train()
    running_loss, running_acc, n = 0.0, 0.0, 0
    for X, y in tqdm(loader, desc="train", leave=False):
        optimizer.zero_grad(set_to_none=True)
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        bsz = y.size(0)
        running_loss += loss.item() * bsz
        running_acc += accuracy(logits.detach(), y) * bsz
        n += bsz
    return running_loss / n, running_acc / n


def evaluate(model, loader, criterion):
    model.eval()
    running_loss, running_acc, n = 0.0, 0.0, 0
    with torch.inference_mode():
        for X, y in tqdm(loader, desc="eval", leave=False):
            logits = model(X)
            loss = criterion(logits, y)
            bsz = y.size(0)
            running_loss += loss.item() * bsz
            running_acc += accuracy(logits, y) * bsz
            n += bsz
    return running_loss / n, running_acc / n


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--epochs', type=int, default=5)
    parser.add_argument('--batch_size', type=int, default=128)
    parser.add_argument('--lr', type=float, default=5e-4)
    parser.add_argument('--weight_decay', type=float, default=0.01)
    parser.add_argument('--d_model', type=int, default=256)
    parser.add_argument('--nhead', type=int, default=4)
    parser.add_argument('--nlayers', type=int, default=4)
    parser.add_argument('--ff', type=int, default=512)
    parser.add_argument('--dropout', type=float, default=0.04)
    parser.add_argument('--min_freq', type=int, default=2)
    parser.add_argument('--seed', type=int, default=42)
    parser.add_argument('--save_path', type=str, default='checkpoint.pt')
    parser.add_argument('--use_quantization', type=bool, default=True)
    parser.add_argument('--use_sparse_pattern', type=bool, default=True)
    parser.add_argument('--sparsity_level', type=float, default=0.1)
    # Parse known arguments, ignoring the rest
    args, unknown = parser.parse_known_args()

    set_seed(args.seed)
    device = get_device()
    print(f"Using device: {device}")

    train_loader, test_loader, vocab, pad_idx = get_dataloaders(batch_size=args.batch_size, device=device, min_freq=args.min_freq)
    vocab_size = len(vocab)
    print(f"Vocab size: {vocab_size}")

    # Disable quantization if using CUDA
    use_quantization = args.use_quantization and (device == torch.device("cpu"))

    model = TransformerTextClassifier(
        vocab_size=vocab_size,
        num_classes=4,
        d_model=args.d_model,
        nhead=args.nhead,
        dim_feedforward=args.ff,
        nlayers=args.nlayers,
        dropout=args.dropout,
        pad_idx=pad_idx,
        use_quantization=use_quantization # Use the updated use_quantization variable
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    steps_per_epoch = max(1, len(train_loader))
    scheduler = OneCycleLR(optimizer, max_lr=args.lr, epochs=args.epochs, steps_per_epoch=steps_per_epoch)

    best_acc = 0.0
    for epoch in range(1, args.epochs + 1):
        # Use a single-line f-string to avoid accidental newline breakage in some editors
        print(f"Epoch {epoch}/{args.epochs}")
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, scheduler)
        te_loss, te_acc = evaluate(model, test_loader, criterion)
        print(f"Train loss {tr_loss:.4f} | acc {tr_acc:.4f} || Test loss {te_loss:.4f} | acc {te_acc:.4f}")
        if te_acc > best_acc:
            best_acc = te_acc
            torch.save({
                'model_state_dict': model.state_dict(),
                'vocab': vocab.get_stoi(),
                'pad_idx': pad_idx,
                'args': vars(args)
            }, args.save_path)
            print(f"Saved new best checkpoint to {args.save_path}")

    print(f"Best test acc: {best_acc:.4f}")


if __name__ == '__main__':
    main()

Using device: cuda
Vocab size: 44136
Epoch 1/5


Train loss 0.7382 | acc 0.7013 || Test loss 0.4107 | acc 0.8550
Saved new best checkpoint to checkpoint.pt
Epoch 2/5


Train loss 0.3364 | acc 0.8831 || Test loss 0.3147 | acc 0.8900
Saved new best checkpoint to checkpoint.pt
Epoch 3/5


Train loss 0.2302 | acc 0.9212 || Test loss 0.2816 | acc 0.9062
Saved new best checkpoint to checkpoint.pt
Epoch 4/5


Train loss 0.1480 | acc 0.9495 || Test loss 0.2539 | acc 0.9192
Saved new best checkpoint to checkpoint.pt
Epoch 5/5


Train loss 0.0838 | acc 0.9726 || Test loss 0.2745 | acc 0.9213
Saved new best checkpoint to checkpoint.pt
Best test acc: 0.9213


# Results

- Using 5 epochs

Baseline 92.09%

Adding Quantitization 92.20%

Adding Sparse Pattern (Sparse Level .2) 92.17%

Adding Sparse Pattern (Sparse Level .4) 92.00%

Adding Sparse Pattern (Sparse Level .1) 92.18%

Using Quantitization and Sparse Pattern (Sparse Level .1) 92.14%

- All results besides a Sparse Level of .4 were an improvement from the baseline result.   Adding both adjustments did not result in a better result.
Individually quantization or the sparse pattern resulted in a slightly better result.